# 05. Monitoring: drift and age-band report, CloudWatch, Clarify and clean-up

1. **Drift and age-band report (BP-802):** compare captured live traffic with the training data, and check selection rates per age band.
2. **CloudWatch (BP-803):** publish the headline numbers as custom metrics, with an alarm and a dashboard.
3. **Clarify and Model Monitor (BP-804):** try SageMaker's managed bias report and monitoring schedule, and record what the lab allows.
4. **Clean-up (BP-805):** delete the endpoint and everything that bills by the hour. It is **off** until `CLEAN_UP = True`.

Run notebook 04 first: it deploys the endpoint and creates the capture files.

In [ ]:
# Setup: repository root, config, lab connection and the state saved by earlier notebooks.
from pathlib import Path

import pandas as pd
import sagemaker

from bankpulse_core.aws import lab_boto_session, read_state, s3_uri, update_state
from bankpulse_core.config import load_config

ROOT = Path.cwd() if (Path.cwd() / 'config').is_dir() else Path.cwd().parent
config = load_config(ROOT / 'config' / 'data_prep.yaml')
STATE = Path(config['output']['processed_dir']) / 'sagemaker_state.json'
state = read_state(STATE)

boto_session = lab_boto_session()                       # bankpulse-lab profile; no keys in code
sm_session = sagemaker.Session(boto_session=boto_session)
sm_client = boto_session.client('sagemaker')
bucket, prefix, role = state['bucket'], state['prefix'], state['role']

import json
import time

from bankpulse_core.io import list_objects, read_csv, read_json, upload_file, write_json
from bankpulse_core.monitoring import build_report, group_from_one_hot, parse_capture

assert state.get('capture_files'), 'Run notebook 04 first: it creates the capture files'
ENDPOINT = state['endpoint']
schema = read_json(state['s3']['feature_schema'])
threshold = float(state['threshold'])
print('endpoint:', ENDPOINT)

## BP-802. Drift and age-band report

- **Reference:** what the model was trained on (the training features) and the scores it gave on validation.
- **Current:** what the endpoint actually received and returned, read from **Data Capture**.
- **PSI** (Population Stability Index) per feature: below 0.1 is stable, 0.1-0.2 is worth watching, above 0.2 means significant drift.
- **Age bands:** the share of customers the model would call in each band; a min/max ratio below 0.8 flags a fairness review.

**Screenshot:** the report tables and chart, and the report JSON in `monitoring-reports/`.

In [ ]:
# Step 1: reference data from the data lake (train.csv without its label column).
reference = read_csv(state['s3']['train'], header=None).iloc[:, 1:]
reference.columns = schema['columns']
from bankpulse_core.evaluation import parse_batch_output
reference_scores = parse_batch_output(state['batch_predictions']['validation'])

# Step 2: live data, from every capture file the endpoint has written so far.
capture_files = [uri for uri in list_objects(state['capture_uri'] + f'/{ENDPOINT}/') if uri.endswith('.jsonl')]
current, current_scores = parse_capture(capture_files, schema['columns'])
print(len(current), 'captured customers from', len(capture_files), 'capture files')

# Step 3: one report with data drift, prediction drift and age-band selection rates.
report = build_report(reference, current, reference_scores, current_scores,
                      groups=group_from_one_hot(current, 'age_band'), threshold=threshold)
display(pd.Series(report['metrics']).to_frame('headline metric'))
print('alerts:', report['alerts'] or 'none')

In [ ]:
import matplotlib.pyplot as plt

top = pd.Series(report['data_drift']['psi']).head(12)[::-1]
ax = top.plot.barh(figsize=(9, 5), color=['#C0392B' if v > 0.2 else '#E07B00' if v > 0.1 else '#2E8B57' for v in top],
                   title='Feature drift: PSI, live traffic vs training data (top 12)')
ax.axvline(0.2, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('PSI (dashed line: 0.2 alert level)')
plt.tight_layout()

display(pd.DataFrame(report['subgroups']['groups']).T.rename_axis('age band'))
print('selection-rate ratio (min/max):', report['subgroups']['min_max_ratio'], '->', report['subgroups']['status'])

In [ ]:
# Step 4: save the report in the data lake (dated) and locally (committed: no account details).
stamp = time.strftime('%Y-%m-%dT%H-%M-%S')
report_path = write_json(report | {'endpoint_kind': 'real-time', 'capture_files': len(capture_files)},
                         ROOT / 'reports' / 'monitoring_report.json')
report_uri = upload_file(report_path, s3_uri(bucket, prefix, 'monitoring-reports', stamp, 'monitoring_report.json'))
state = update_state(STATE, monitoring_report=report_uri)
print('saved', report_uri.split(f'{bucket}/', 1)[1])

## BP-803. CloudWatch metrics, alarm and dashboard

- **Custom metrics:** the report's four headline numbers, published under the namespace `BankPulse/ModelMonitoring` with the endpoint name as a dimension.
- **Alarm:** goes to `ALARM` when the maximum feature PSI is above 0.2. A new alarm shows `INSUFFICIENT_DATA` until CloudWatch has evaluated it (a few minutes).
- **Dashboard:** one page with the four metrics.

**Screenshots:** the metrics graph, the alarm and its state, and the dashboard.

In [ ]:
cloudwatch = boto_session.client('cloudwatch')
NAMESPACE = 'BankPulse/ModelMonitoring'
dimensions = [{'Name': 'EndpointName', 'Value': ENDPOINT}]

# Step 1: one data point per headline metric.
cloudwatch.put_metric_data(Namespace=NAMESPACE, MetricData=[
    {'MetricName': name, 'Dimensions': dimensions, 'Value': float(value), 'Unit': 'None'}
    for name, value in report['metrics'].items()
])

# Step 2: alarm on feature drift.
ALARM = f'{ENDPOINT}-feature-drift'
cloudwatch.put_metric_alarm(
    AlarmName=ALARM, AlarmDescription='Max feature PSI above 0.2: live data has drifted from the training data',
    Namespace=NAMESPACE, MetricName='max_feature_psi', Dimensions=dimensions,
    Statistic='Maximum', Period=300, EvaluationPeriods=1, Threshold=0.2,
    ComparisonOperator='GreaterThanThreshold', TreatMissingData='notBreaching',
)

# Step 3: dashboard with the four metrics.
region = sm_session.boto_region_name
widgets = [{'type': 'metric', 'x': (i % 2) * 12, 'y': (i // 2) * 6, 'width': 12, 'height': 6,
            'properties': {'title': name, 'region': region, 'stat': 'Maximum', 'period': 300, 'view': 'singleValue',
                           'metrics': [[NAMESPACE, name, 'EndpointName', ENDPOINT]]}}
           for i, name in enumerate(report['metrics'])]
DASHBOARD = 'bankpulse-model-monitoring'
cloudwatch.put_dashboard(DashboardName=DASHBOARD, DashboardBody=json.dumps({'widgets': widgets}))
state = update_state(STATE, cloudwatch={'namespace': NAMESPACE, 'alarm': ALARM, 'dashboard': DASHBOARD})

alarm = cloudwatch.describe_alarms(AlarmNames=[ALARM])['MetricAlarms'][0]
print('metrics published to', NAMESPACE)
print('alarm', ALARM, 'state:', alarm['StateValue'])
print('dashboard:', DASHBOARD)

## BP-804. Clarify and Model Monitor (what the lab allows)

Two managed SageMaker features, each tried and recorded. AWS Academy labs block some of them, so a failure is a result too.
- **SageMaker Clarify, pre-training bias:** a processing job that measures label imbalance between age 60+ and everyone else in the validation data (`CI` and `DPL` metrics).
- **Model Monitor:** a baseline job that learns statistics and constraints from the training data, then an **hourly monitoring schedule** on the endpoint's captured data.

Both jobs run in parallel (about 6-10 minutes).

**Screenshot:** this section's results table, plus the Clarify report and the monitoring schedule if they were created.

In [ ]:
from sagemaker.clarify import BiasConfig, DataConfig, SageMakerClarifyProcessor
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

attempts = {}
headers = ['label'] + schema['columns']

# Clarify: pre-training bias on the validation file (label first, no header).
try:
    clarify = SageMakerClarifyProcessor(role=role, instance_count=1, instance_type='ml.m5.large',
                                        sagemaker_session=sm_session)
    clarify.run_pre_training_bias(
        data_config=DataConfig(s3_data_input_path=state['s3']['validation'], headers=headers, label='label',
                               dataset_type='text/csv', s3_output_path=s3_uri(bucket, prefix, 'clarify', 'pre-training')),
        data_bias_config=BiasConfig(label_values_or_threshold=[1], facet_name='age_band__60plus',
                                    facet_values_or_threshold=[1]),
        methods=['CI', 'DPL'], wait=False, logs=False,
    )
    attempts['clarify'] = {'job': clarify.latest_job.job_name, 'started': True}
except Exception as error:                      # record whatever the lab says, then carry on
    attempts['clarify'] = {'started': False, 'error': f'{type(error).__name__}: {str(error)[:200]}'}

# Model Monitor: baseline statistics and constraints from the training features.
try:
    monitor = DefaultModelMonitor(role=role, instance_count=1, instance_type='ml.m5.large', volume_size_in_gb=20,
                                  max_runtime_in_seconds=1800, sagemaker_session=sm_session)
    monitor.suggest_baseline(baseline_dataset=state['s3']['validation_features'],
                             dataset_format=DatasetFormat.csv(header=False),
                             output_s3_uri=s3_uri(bucket, prefix, 'model-monitor', 'baseline'), wait=False, logs=False)
    attempts['model_monitor_baseline'] = {'job': monitor.latest_baselining_job_name, 'started': True}
except Exception as error:
    attempts['model_monitor_baseline'] = {'started': False, 'error': f'{type(error).__name__}: {str(error)[:200]}'}

display(pd.DataFrame(attempts).T)

In [ ]:
# Wait for whichever jobs started, then create the monitoring schedule if the baseline worked.
if attempts['clarify'].get('started'):
    try:
        clarify.latest_job.wait(logs=False)
        attempts['clarify']['result'] = 'Completed: report in ' + s3_uri(bucket, prefix, 'clarify', 'pre-training')
    except Exception as error:
        attempts['clarify']['result'] = f'{type(error).__name__}: {str(error)[:200]}'

if attempts['model_monitor_baseline'].get('started'):
    try:
        monitor.latest_baselining_job.wait(logs=False)
        attempts['model_monitor_baseline']['result'] = 'Completed: statistics and constraints in ' + s3_uri(
            bucket, prefix, 'model-monitor', 'baseline')
    except Exception as error:
        attempts['model_monitor_baseline']['result'] = f'{type(error).__name__}: {str(error)[:200]}'

if str(attempts['model_monitor_baseline'].get('result', '')).startswith('Completed'):
    try:
        SCHEDULE = f'{ENDPOINT}-data-quality'[:63]
        monitor.create_monitoring_schedule(
            monitor_schedule_name=SCHEDULE, endpoint_input=ENDPOINT,
            output_s3_uri=s3_uri(bucket, prefix, 'model-monitor', 'reports'),
            statistics=monitor.baseline_statistics(), constraints=monitor.suggested_constraints(),
            schedule_cron_expression='cron(0 * ? * * *)',      # hourly
        )
        attempts['monitoring_schedule'] = {'started': True, 'result': f'created {SCHEDULE} (hourly)'}
        state = update_state(STATE, monitoring_schedule=SCHEDULE)
    except Exception as error:
        attempts['monitoring_schedule'] = {'started': False, 'error': f'{type(error).__name__}: {str(error)[:200]}'}

write_json(attempts, ROOT / 'reports' / 'managed_monitoring_attempts.json')
display(pd.DataFrame(attempts).T)

## BP-805. Clean-up

These resources bill **by the hour** until deleted: the monitoring schedule (an hourly job), the endpoint, its configuration and model. S3 objects, the registry, the Model Card and CloudWatch cost nothing (or next to nothing) and stay as the project's record.

Take your screenshots first, then set `CLEAN_UP = True` and run the cell. Afterwards, click **End Lab** in AWS Academy.

**Screenshot:** the SageMaker endpoints list, empty.

In [ ]:
CLEAN_UP = False          # set to True after taking the screenshots

if CLEAN_UP:
    state = read_state(STATE)
    # Step 1: the monitoring schedule first (it reads from the endpoint).
    if state.get('monitoring_schedule'):
        try:
            sm_client.delete_monitoring_schedule(MonitoringScheduleName=state['monitoring_schedule'])
            time.sleep(30)                     # the schedule must be gone before the endpoint is deleted
        except sm_client.exceptions.ResourceNotFound:
            pass
    # Step 2: endpoint, then its configuration, then the model behind it.
    endpoint = sm_client.describe_endpoint(EndpointName=ENDPOINT)
    config_name = endpoint['EndpointConfigName']
    model_names = [v['ModelName'] for v in sm_client.describe_endpoint_config(
        EndpointConfigName=config_name)['ProductionVariants']]
    sm_client.delete_endpoint(EndpointName=ENDPOINT)
    sm_client.delete_endpoint_config(EndpointConfigName=config_name)
    for name in model_names:
        sm_client.delete_model(ModelName=name)
    state = update_state(STATE, cleaned_up=time.strftime('%Y-%m-%dT%H:%M:%S'))
    print('deleted endpoint, endpoint config and model(s):', ENDPOINT, config_name, model_names)

running = [e['EndpointName'] for e in sm_client.list_endpoints()['Endpoints']]
print('endpoints still running:', running or 'none')